In [39]:
import pandas as pd
from pathlib import Path
# import kagglehub
import os

# **Define reusable functions**

In [40]:
def print_line(num):
    print('=' * num)
    
def print_txt(text, num=50):
    print_line(num)
    print(text)
    print_line(num)

In [41]:
# uncomment to use with kaggle
# clean_data_path = kagglehub.dataset_download("karapy/azure-1-processed")

# print("Path to dataset files:", clean_data_path)

In [42]:
# uncomment to use with kaggle
# CLEAN_DATA = Path.joinpath(Path(clean_data_path), 'cleaned_dataset-electric.csv')

RAW_DATA_DIR = '../data/raw/dataset-electric.csv'
SAVE_DATA_DIR = Path('../data/processed/')
CLEAN_DATA = 'cleaned_dataset-electric.csv'
FEATURED_DATA_DIR = Path('../data/featured')

FEATURED_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [43]:
# uncomment to use with kaggle
# clean_df = pd.read_csv(CLEAN_DATA)

clean_df = pd.read_csv(Path.joinpath(SAVE_DATA_DIR, CLEAN_DATA))

# **Feature Engineering**

In [44]:

clean_df['Datetime'] = pd.to_datetime(clean_df['Datetime'])

In [45]:
print_txt('extracts hour, day, and year from each timestamp "Datetime"')

clean_df['DayOfWeek'] = clean_df['Datetime'].dt.dayofweek
clean_df['Month'] = clean_df['Datetime'].dt.month
clean_df['Year'] = clean_df['Datetime'].dt.year
clean_df['Hour'] = clean_df['Datetime'].dt.hour
print('Done')

extracts hour, day, and year from each timestamp "Datetime"
Done


In [46]:
# create a column to checks whether the day is Saturday or Sunday
clean_df['IsWeekend'] = clean_df['DayOfWeek'].isin([5, 6]).astype(int)

In [47]:
# create a column consumption 24 hours ago
clean_df['lag_24'] = clean_df['PJME_MW'].shift(24)

# create a column consumption 168 hours ago
clean_df['lag_168'] = clean_df['PJME_MW'].shift(168)

# Calculate the average electricity consumption over the previous 24 hours,
clean_df['roll_24'] = (clean_df['PJME_MW'].shift(1).rolling(24) .mean())

In [48]:
print_txt('Check the missing values in the 3 feature columns')
display(clean_df.loc[clean_df[['lag_24', 'lag_168', 'roll_24']].isna().any(axis=1)])

Check the missing values in the 3 feature columns


,Datetime,PJME_MW,DayOfWeek,Month,Year,Hour,IsWeekend,lag_24,lag_168,roll_24
0,2002-01-01 01:00:00,30393.0,1,1,2002,1,0,NaN,NaN,NaN
1,2002-01-01 02:00:00,29265.0,1,1,2002,2,0,NaN,NaN,NaN
2,2002-01-01 03:00:00,28357.0,1,1,2002,3,0,NaN,NaN,NaN
3,2002-01-01 04:00:00,27899.0,1,1,2002,4,0,NaN,NaN,NaN
4,2002-01-01 05:00:00,28057.0,1,1,2002,5,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
163,2002-01-07 20:00:00,38780.0,0,1,2002,20,0,35306.0,NaN,32786.541667
164,2002-01-07 21:00:00,37777.0,0,1,2002,21,0,34315.0,NaN,32931.291667
165,2002-01-07 22:00:00,35997.0,0,1,2002,22,0,32731.0,NaN,33075.541667
166,2002-01-07 23:00:00,33579.0,0,1,2002,23,0,30646.0,NaN,33211.625000


In [49]:
print_txt('Drop roll_24, lag_24 ,and lad_168 rows with NaN values ')
clean_df = clean_df.dropna(subset=['lag_24', 'lag_168', 'roll_24'] ).reset_index(drop=True)

Drop roll_24, lag_24 ,and lad_168 rows with NaN values 


In [50]:
print_txt('New data sample')
display(clean_df.head(10))

New data sample


,Datetime,PJME_MW,DayOfWeek,Month,Year,Hour,IsWeekend,lag_24,lag_168,roll_24
0,2002-01-08 01:00:00,29445.0,1,1,2002,1,0,26862.0,30393.0,33452.583333
1,2002-01-08 02:00:00,28670.0,1,1,2002,2,0,25976.0,29265.0,33560.208333
2,2002-01-08 03:00:00,28375.0,1,1,2002,3,0,25641.0,28357.0,33672.458333
3,2002-01-08 04:00:00,28542.0,1,1,2002,4,0,25666.0,27899.0,33786.375000
4,2002-01-08 05:00:00,29261.0,1,1,2002,5,0,26328.0,28057.0,33906.208333
5,2002-01-08 06:00:00,31348.0,1,1,2002,6,0,28267.0,28654.0,34028.416667
6,2002-01-08 07:00:00,35335.0,1,1,2002,7,0,31938.0,29308.0,34156.791667
7,2002-01-08 08:00:00,37841.0,1,1,2002,8,0,34678.0,29595.0,34298.333333
8,2002-01-08 09:00:00,37417.0,1,1,2002,9,0,35349.0,29943.0,34430.125000
9,2002-01-08 10:00:00,36824.0,1,1,2002,10,0,35542.0,30692.0,34516.291667


In [51]:

clean_df.to_csv(FEATURED_DATA_DIR / 'featured_dataset-electric.csv',index=False )